![Impressions](https://PixelServer20190423114238.azurewebsites.net/api/impressions/NotebookVM/tutorials/quickstart-ci/AzureMLin10mins.png)

# **Sentiment analysis - Azure Deployment**

**Import librairies**

In [2]:
import os

import azureml.core
from azureml.core import Workspace, Dataset

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import glob
import re

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin

from tensorflow import keras
import tensorflow as tf

from keras.preprocessing.text import Tokenizer
from keras.preprocessing.sequence import pad_sequences
from keras.models import Sequential
from keras.layers import Dense
from keras.layers import Embedding
from keras.layers import Flatten
from keras.layers import LSTM
from keras.wrappers.scikit_learn import KerasClassifier


**Import datas**

In [3]:
ws = Workspace.from_config()
ds1 = ws.datasets['sentiment_1600']

df = ds1.to_pandas_dataframe()

x_train_pipe = df['text']
y_train_pipe = np.array(df['label'].map({0:0, 4:1})).reshape(-1,1)


**Define pipeline functions**

In [5]:

 
def preprocess_pipe(text):
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'@\S+', '', text)
    text = re.sub('[^a-zA-Z0-9\s]','',text)
    text = str(text).lower()
    return text

class TokenizerTransformer(BaseEstimator, TransformerMixin, Tokenizer):

    def __init__(self, **tokenizer_params):
        Tokenizer.__init__(self, **tokenizer_params)

    def fit(self, X, y=None):
        self.fit_on_texts(X)
        return self

    def transform(self, X, y=None):
        X_transformed = self.texts_to_sequences(X)
        return X_transformed
  
    
class PadSequencesTransformer(BaseEstimator, TransformerMixin):

    def __init__(self, maxlen):
        self.maxlen = maxlen

    def fit(self, X, y=None):
        return self

    def transform(self, X, y=None):
        X_padded = pad_sequences(X, maxlen=self.maxlen)
        return X_padded    
    
class PreprocessTransformer(BaseEstimator, TransformerMixin):

    def __init__(self):
        None

    def fit(self, X, y=None):
        return self

    def transform(self, X, y=None):
        X_to_preprocess = pd.DataFrame(X)
        X_preprocessed = X_to_preprocess['text'].apply(lambda x: preprocess_pipe(x))
        return X_preprocessed    
    
def create_model(embedding):
    dropout = 0.5
    model = Sequential()
    model.add(embedding)
    model.add(LSTM(100, dropout=dropout, recurrent_dropout=dropout))
    model.add(Dense(1, activation='sigmoid'))
    model.compile(loss = 'binary_crossentropy', optimizer='adam', metrics = ['accuracy'])
    
    return model  

def glove_embedding_layer(glove, tokenizer):

    vocab_size = len(tokenizer.word_index) + 1 
    embedding_matrix_glove = np.zeros((vocab_size, 100))

    for word, i in tokenizer.word_index.items():
        embedding_vector = glove.get(word)
        if embedding_vector is not None:
            embedding_matrix_glove[i] = embedding_vector

    embedding_vector_length = 100
    input_length = 100
    embedding_glove = Embedding(
        vocab_size,
        embedding_vector_length,
        weights=[embedding_matrix_glove],
        input_length=input_length,
        trainable=False) 
    
    return embedding_glove      


**Import glove vectors**

In [6]:
ds_glove = ws.datasets['glove']

import tempfile
mounted_path = tempfile.mkdtemp()

# mount dataset onto the mounted_path of a Linux-based compute
mount_context = ds_glove.mount(mounted_path)

mount_context.start()

glove = dict()
with open(mounted_path + '/glove.twitter.27B.100d.txt', 'r') as f:
    content = f.readlines()
    for line in content:
        values = line.split()
        word = values[0]
        coefs = np.asarray(values[1:], dtype='float32')
        glove[word] = coefs




Downloaded path: /tmp/tmp0fiid04a/aaa518da-a192-418f-9356-9bab9c29afe4/UI/02-14-2022_100915_UTC/glove.twitter.27B.100d.txt is different from target path: /tmp/tmp0fiid04a/aaa518da-a192-418f-9356-9bab9c29afe4/glove.twitter.27B.100d.txt


**Create pipeline**

In [7]:
preproc = PreprocessTransformer()
my_tokenizer = TokenizerTransformer()
my_padder = PadSequencesTransformer(maxlen=100)
embedding_glove_pipe = glove_embedding_layer(glove, my_tokenizer.fit(preproc.transform(x_train_pipe)))

my_model = KerasClassifier(
               build_fn=create_model, 
               epochs=5,
               embedding=embedding_glove_pipe
)

pipeline = Pipeline([
              ('preprocess', preproc),
              ('tokenizer', my_tokenizer),
              ('padder', my_padder),
              ('model', my_model)
])



<ipython-input-7-6b2964547509>:6: DeprecationWarning: KerasClassifier is deprecated, use Sci-Keras (https://github.com/adriangb/scikeras) instead.
  my_model = KerasClassifier(


**Train pipeline**

In [8]:
import mlflow

# create experiment and start logging to a new run in the experiment
experiment_name = "sentiment_analysis_pipeline"

# set up MLflow to track the metrics
mlflow.set_tracking_uri(ws.get_mlflow_tracking_uri())

mlflow.set_experiment(experiment_name)
mlflow.sklearn.autolog()

# train the model
with mlflow.start_run() as run:
    pipeline.fit(x_train_pipe, y_train_pipe)



2022/02/15 23:44:52 INFO mlflow.tracking.fluent: Experiment with name 'sentiment_analysis_pipeline' does not exist. Creating a new experiment.
2022/02/15 23:44:53 WARNING mlflow.utils.autologging_utils: You are using an unsupported version of sklearn. If you encounter errors during autologging, try upgrading / downgrading sklearn to a supported version, or try upgrading MLflow.


Epoch 1/5
50/50 [==============================] - 7s 86ms/step - loss: 0.6383 - accuracy: 0.6456
Epoch 2/5
50/50 [==============================] - 4s 85ms/step - loss: 0.5776 - accuracy: 0.6975
Epoch 3/5
50/50 [==============================] - 4s 84ms/step - loss: 0.5604 - accuracy: 0.7125
Epoch 4/5
50/50 [==============================] - 4s 84ms/step - loss: 0.5520 - accuracy: 0.7231
Epoch 5/5
50/50 [==============================] - 1s 18ms/step - loss: 0.4916 - accuracy: 0.7675
INFO:tensorflow:Assets written to: ram://b7ab7e02-77f0-4e5b-b416-742d4cb34aa5/assets


**Register pipeline**

In [9]:
# register the model
model_uri = "runs:/{}/model".format(run.info.run_id)
model = mlflow.register_model(model_uri, "sentiment_analysis_pipeline")


Successfully registered model 'sentiment_analysis_pipeline'.
2022/02/15 23:46:14 INFO mlflow.tracking._model_registry.client: Waiting up to 300 seconds for model version to finish creation.                     Model name: sentiment_analysis_pipeline, version 1
Created version '1' of model 'sentiment_analysis_pipeline'.


**Create environment and config deployment**

In [10]:
# create environment for the deploy
from azureml.core.environment import Environment
from azureml.core.conda_dependencies import CondaDependencies
from azureml.core.webservice import AciWebservice

env = Environment.from_conda_specification(name='dependencies_env', file_path="conda_dependencies.yml")

# create deployment config i.e. compute resources
aciconfig = AciWebservice.deploy_configuration(
    cpu_cores=1,
    memory_gb=1,
    tags={"data": "sentiment", "method": "lstm"},
    description="Predict sentiment with lstm glove",
)

**Deploy pipeline**

In [12]:
%%time
import uuid
from azureml.core.model import InferenceConfig
from azureml.core.environment import Environment
from azureml.core.model import Model

# get the registered model
model = Model(ws, "sentiment_analysis_pipeline")

# create an inference config i.e. the scoring script and environment
inference_config = InferenceConfig(entry_script="score.py", environment=env)

# deploy the service
service_name = "sentiment-analysis-" + str(uuid.uuid4())[:4]
service = Model.deploy(
    workspace=ws,
    name=service_name,
    models=[model],
    inference_config=inference_config,
    deployment_config=aciconfig,
)

service.wait_for_deployment(show_output=True)

Tips: You can try get_logs(): https://aka.ms/debugimage#dockerlog or local deployment: https://aka.ms/debugimage#debug-locally to debug if deployment takes longer than 10 minutes.
Running
2022-02-15 23:48:17+00:00 Creating Container Registry if not exists.
2022-02-15 23:48:17+00:00 Registering the environment.
2022-02-15 23:48:19+00:00 Use the existing image.
2022-02-15 23:48:19+00:00 Generating deployment configuration.
2022-02-15 23:48:21+00:00 Submitting deployment to compute.
2022-02-15 23:48:26+00:00 Checking the status of deployment sentiment-analysis-eb46..
2022-02-15 23:50:30+00:00 Checking the status of inference endpoint sentiment-analysis-eb46.
Succeeded
ACI service creation operation finished, operation "Succeeded"
CPU times: user 480 ms, sys: 88.1 ms, total: 568 ms
Wall time: 2min 18s
